In [1]:
import pandas as pd
import numpy as np
import os
import glob
from osgeo import ogr, osr
from osgeo import gdal
from tqdm import tqdm
from shapely.ops import unary_union
from shapely.wkb import loads
from shapely.geometry import Polygon, MultiPolygon
import gc
import time

from helpers_merging import *

gdal.UseExceptions()

In [ ]:
############################
# set paths

# path to sample fields
path_fields = '../sample_polygons/'

# path to vector dataset with overlapping tile extents
path_tile = '../sample_polygons/extents.gpkg'

# path to vector dataset only representing the border lines of the tile extents in path_tile; buffered by 1 m
path_tile_buff = '../sample_polygons/extents_border_1m_buffer.gpkg'

# path to output folder
output_folder_path = '../sample_polygons/'
os.makedirs(output_folder_path, exist_ok=True)

check_large_fields = False

In [21]:
##############################
# read datasets and set processing variables

# set driver
driver_gpkg = ogr.GetDriverByName("GPKG")

# set epsg
epsg = 32636
feat_crs = osr.SpatialReference()
feat_crs.ImportFromEPSG(epsg)

# read tile datasets
ds_tile = driver_gpkg.Open(path_tile)
tile_lyr = ds_tile.GetLayer()

ds_buff = driver_gpkg.Open(path_tile_buff)
buffer_lyr = ds_buff.GetLayer()
buffCRS = buffer_lyr.GetSpatialRef()

# set ti/tj of center tile
# in final implementation this needs to be adjusted with a moving window logic if there are multiple images
ti = 1
tj = 1

out_file = output_folder_path + f"/mozambique_tile_{ti}_{tj}_cleaned.gpkg"

# create list to store features that intersect both borders
both_intersect_geom = []                  

# create list to store dissolved large fields
dissolved_features = []

# create list to store central complete fields
central_features = []

# create list to store border fields
border_features = []

In [22]:
#############################
# define 3x3 and 5x5 window around the target ti, tj tile

# Please note: the code repository provides example field boundaries only for a 3x3 window.
# The overlap regions of the 3x3 files get cleaned and are written to file with the 3x3 window extent
# In case there are neighboring tiles around the 3x3 window, the code reads in the adjacent tiles (5x5 wndow) 
# and also cleans the overlap areas of the surrounding tiles which overlap with the 3x3 window.
# The 5x5 window check is implemented in the code but ignored if there are no adjacent tiles.

# get list of current and neighboring tile paths  
tmp_border_paths, tmp_core_paths = find_files(path_fields, ti, tj)

# get larger tile geom (3x3)
large_tile_geom = union_to_large_tile(tile_lyr, tmp_core_paths, epsg)

In [23]:
#############################
# collect all fields from core window (3x3)

# open datasources of the tiles except if tile not in file list
for core_path in tmp_core_paths:
    
    # open datasource and get layer
    ds_tmp = driver_gpkg.Open(core_path)
    lyr_tmp = ds_tmp.GetLayer()

    # get layer definition
    lyr_defn = lyr_tmp.GetLayerDefn()
    
    # get GeoTransform
    tmpCRS = lyr_tmp.GetSpatialRef()

    #coordtrans = osr.CoordinateTransformation(buffCRS, tmpCRS)
    #coordtrans_back = osr.CoordinateTransformation(tmpCRS, buffCRS)
    
    # get epsg, ti, tj of current file
    file_name = core_path.split("/")[-1] 
    ti_tmp, tj_tmp = file_name.split("_")[1:3]
    ti_int = int(ti_tmp)
    tj_int = int(tj_tmp)
    epsg_tmp = epsg # in case there are multiple epsg adjust to infer which epsg is currently needed
    
    # set list of buffer filters
    filter_b_ls = [f"ti IN ({ti_int}, {ti_int-1}) AND tj = {tj_int}", # west
                f"ti IN ({ti_int}, {ti_int+1}) AND tj = {tj_int}",    # east
                f"ti = {ti_int} AND tj IN ({tj_int}, {tj_int-1})",    # south
                f"ti = {ti_int} AND tj IN ({tj_int}, {tj_int+1})"]    # north
    
    # set list of tile (not buffered) filters
    filter_t_ls = [f"tile_id = '{epsg_tmp}_{ti_tmp}_{tj_tmp}' OR tile_id = '{epsg_tmp}_{ti_int-1}_{tj_tmp}'", # west
                f"tile_id = '{epsg_tmp}_{ti_tmp}_{tj_tmp}' OR tile_id = '{epsg_tmp}_{ti_int+1}_{tj_tmp}'",    # east
                f"tile_id = '{epsg_tmp}_{ti_tmp}_{tj_tmp}' OR tile_id = '{epsg_tmp}_{ti_tmp}_{tj_int-1}'",    # south
                f"tile_id = '{epsg_tmp}_{ti_tmp}_{tj_tmp}' OR tile_id = '{epsg_tmp}_{ti_tmp}_{tj_int+1}'"]    # north

    # get border area
    border_area = get_border_area(tile_lyr, filter_t_ls, all_four_borders=True)
    #border_area.Transform(coordtrans)

    # get buffer tile
    tile_lyr.SetAttributeFilter(f"tile_id = '{epsg_tmp}_{ti_tmp}_{tj_tmp}'")
    current_tile = tile_lyr.GetNextFeature()
    current_tile_geom = current_tile.GetGeometryRef()

    # filter corresponding buffer tile
    buffer_lyr.SetAttributeFilter(f"ti = '{int(ti_tmp)}' AND tj = '{int(tj_tmp)}'")
    
    # same ti and tj might exist for different CRS -> select correct buffer tile with spatial filter
    buffer_lyr.SetSpatialFilter(current_tile_geom)
    buff_feat = buffer_lyr.GetNextFeature()
    buff_geom = buff_feat.GetGeometryRef()
    #buff_geom.Transform(coordtrans)

    ###################################
    # assign features to border_features or central_features lists for individual processing

    for feat in lyr_tmp:
        geom = feat.GetGeometryRef()

        # filter features that fall within overlap area
        if border_area.Intersects(geom):
            # filter features that are complete (do not touch border buffer line)
            if not buff_geom.Intersects(geom):
                border_features.append(feat.Clone())            

        else:
            # copy central features that do not fall within overlap area
            central_features.append(feat.Clone())

    #border_area.Transform(coordtrans_back)
    #buff_geom.Transform(coordtrans_back)

    tile_lyr.ResetReading()

    #####################################
    # identify fields touching both buffered border lines on all 4 edges (i.e. large fields) if check_large_fields = True
    if check_large_fields:
        for i in range(4):
            
            # set filter to check borders for overlapping large fields
            buffer_lyr.SetAttributeFilter(filter_b_ls[i])
            tile_lyr.SetAttributeFilter(filter_t_ls[i])
                            
            # get relevant border
            # parallel buffer lines
            border_geoms = get_relevant_border(tile_lyr, buffer_lyr)
            
            # loop over fields
            for feat_tmp in lyr_tmp:

                # set to True and overwrite if False
                bool_intersect = True
                geom_tmp = feat_tmp.GetGeometryRef()

                # loop over tile boundaries
                for geom_buff in border_geoms:

                    if not geom_buff.Intersects(geom_tmp):
                        bool_intersect = False
                        break

                if bool_intersect:
                    
                    both_intersect_geom.append(feat_tmp.Clone())

            lyr_tmp.ResetReading()  

    # reset reading and close ds
    lyr_tmp = None
    ds_tmp = None
    tile_lyr.SetAttributeFilter(None)
    buffer_lyr.SetAttributeFilter(None)
    buffer_lyr.SetSpatialFilter(None)

In [24]:
#################################
# add fields from overlap areas of larger window (5x5)

# open datasources of the tiles except if tile not in file list
if tmp_border_paths:
    for border_path in tmp_border_paths:

        # open datasource and get layer
        ds_tmp = driver_gpkg.Open(border_path)
        lyr_tmp = ds_tmp.GetLayer()

        # get layer definion
        lyr_defn = lyr_tmp.GetLayerDefn()
        
        # get GeoTransform
        tmpCRS = lyr_tmp.GetSpatialRef()
        #coordtrans = osr.CoordinateTransformation(buffCRS, tmpCRS)
        #coordtrans_back = osr.CoordinateTransformation(tmpCRS, buffCRS)
        
        # get epsg, ti, tj of current file
        file_name = border_path.split("/")[-1]
        epsg_tmp, ti_tmp, tj_tmp = file_name.split("_")[2:5]
        ti_int = int(ti_tmp)
        tj_int = int(tj_tmp)

        # north west corner
        if ti_int == ti-2 and tj_int == tj+2:
            filter_b_ls = [f"ti IN ({ti_int}, {ti_int+1}) AND tj = {tj_int}", # east
                        f"ti = {ti_int} AND tj IN ({tj_int}, {tj_int-1})"]    # south
            filter_t_ls = [f"tile_id = '{epsg_tmp}_{ti_tmp}_{tj_tmp}' OR tile_id = '{epsg_tmp}_{ti_int+1}_{tj_tmp}'", # east
                        f"tile_id = '{epsg_tmp}_{ti_tmp}_{tj_tmp}' OR tile_id = '{epsg_tmp}_{ti_tmp}_{tj_int-1}'"]    # south
            # get border area
            border_area = get_border_area(tile_lyr, filter_t_ls, all_four_borders=False, corner=True)
            if border_area == None: continue 
            #border_area.Transform(coordtrans)

        # north east corner
        elif ti_int == ti+2 and tj_int == tj+2:
            filter_b_ls = [f"ti IN ({ti_int}, {ti_int-1}) AND tj = {tj_int}", # west
                        f"ti = {ti_int} AND tj IN ({tj_int}, {tj_int-1})"]    # south
            filter_t_ls = [f"tile_id = '{epsg_tmp}_{ti_tmp}_{tj_tmp}' OR tile_id = '{epsg_tmp}_{ti_int-1}_{tj_tmp}'", # west
                        f"tile_id = '{epsg_tmp}_{ti_tmp}_{tj_tmp}' OR tile_id = '{epsg_tmp}_{ti_tmp}_{tj_int-1}'"]    # south
            # get border area
            border_area = get_border_area(tile_lyr, filter_t_ls, all_four_borders=False, corner=True)
            if border_area == None: continue
            #border_area.Transform(coordtrans)

        # south west corner
        elif ti_int == ti-2 and tj_int == tj-2:
            filter_b_ls = [f"ti = {ti_int} AND tj IN ({tj_int}, {tj_int+1})", # north
                        f"ti IN ({ti_int}, {ti_int+1}) AND tj = {tj_int}"]    # east
            filter_t_ls = [f"tile_id = '{epsg_tmp}_{ti_tmp}_{tj_tmp}' OR tile_id = '{epsg_tmp}_{ti_tmp}_{tj_int+1}'", # north
                        f"tile_id = '{epsg_tmp}_{ti_tmp}_{tj_tmp}' OR tile_id = '{epsg_tmp}_{ti_int+1}_{tj_tmp}'"]    # east
            # get border area
            border_area = get_border_area(tile_lyr, filter_t_ls, all_four_borders=False, corner=True)
            if border_area == None: continue
            #border_area.Transform(coordtrans)

        # south east corner
        elif ti_int == ti+2 and tj_int == tj-2:
            filter_b_ls = [f"ti = {ti_int} AND tj IN ({tj_int}, {tj_int+1})", # north
                        f"ti IN ({ti_int}, {ti_int-1}) AND tj = {tj_int}"]    # west
            filter_t_ls = [f"tile_id = '{epsg_tmp}_{ti_tmp}_{tj_tmp}' OR tile_id = '{epsg_tmp}_{ti_tmp}_{tj_int+1}'", # north
                        f"tile_id = '{epsg_tmp}_{ti_tmp}_{tj_tmp}' OR tile_id = '{epsg_tmp}_{ti_int-1}_{tj_tmp}'"]    # west
            # get border area
            border_area = get_border_area(tile_lyr, filter_t_ls, all_four_borders=False, corner=True)
            if border_area == None: continue 
            #border_area.Transform(coordtrans)

        # west border tiles
        elif ti_int == ti-2:
            filter_b_ls = [f"ti IN ({ti_int}, {ti_int+1}) AND tj = {tj_int}"] # east
            filter_t_ls = [f"tile_id = '{epsg_tmp}_{ti_tmp}_{tj_tmp}' OR tile_id = '{epsg_tmp}_{ti_int+1}_{tj_tmp}'"] # east
            # get border area
            border_area = get_border_area(tile_lyr, filter_t_ls, all_four_borders=False)
            if border_area == None: continue 
            #border_area.Transform(coordtrans)

        # east border tiles:
        elif ti_int == ti+2:
            filter_b_ls = [f"ti IN ({ti_int}, {ti_int-1}) AND tj = {tj_int}"] # west
            filter_t_ls = [f"tile_id = '{epsg_tmp}_{ti_tmp}_{tj_tmp}' OR tile_id = '{epsg_tmp}_{ti_int-1}_{tj_tmp}'"] # west
            # get border area
            border_area = get_border_area(tile_lyr, filter_t_ls, all_four_borders=False)
            if border_area == None: continue
            #border_area.Transform(coordtrans)

        # north border tiles
        elif tj_int == tj+2:
            filter_b_ls = [f"ti = {ti_int} AND tj IN ({tj_int}, {tj_int-1})"] # south
            filter_t_ls = [f"tile_id = '{epsg_tmp}_{ti_tmp}_{tj_tmp}' OR tile_id = '{epsg_tmp}_{ti_tmp}_{tj_int-1}'"] # south
            # get border area
            border_area = get_border_area(tile_lyr, filter_t_ls, all_four_borders=False)
            if border_area == None: continue 
            #border_area.Transform(coordtrans)

        # south border tiles
        elif tj_int == tj-2:
            filter_b_ls = [f"ti = {ti_int} AND tj IN ({tj_int}, {tj_int+1})"] # north
            filter_t_ls = [f"tile_id = '{epsg_tmp}_{ti_tmp}_{tj_tmp}' OR tile_id = '{epsg_tmp}_{ti_tmp}_{tj_int+1}'"] # north
            # get border area (i.e. overlap area of current tile and neighboring tiles)
            border_area = get_border_area(tile_lyr, filter_t_ls, all_four_borders=False)
            if border_area == None: continue 
            #border_area.Transform(coordtrans)

        else:
            print(f"no corresponding border found for {file_name}")

        # get tile
        tile_lyr.SetAttributeFilter(f"tile_id = '{epsg_tmp}_{ti_tmp}_{tj_tmp}'")
        current_tile = tile_lyr.GetNextFeature()
        current_tile_geom = current_tile.GetGeometryRef()
        # filter corresponding tile
        buffer_lyr.SetAttributeFilter(f"ti = '{int(ti_tmp)}' AND tj = '{int(tj_tmp)}'")
        # same ti and tj might exist for different CRS -> select correct tile with spatial filter
        buffer_lyr.SetSpatialFilter(current_tile_geom)
        buff_feat = buffer_lyr.GetNextFeature()
        buff_geom = buff_feat.GetGeometryRef()
        #buff_geom.Transform(coordtrans)   
        
        ###########################################
        # assign features to border_features or central_features lists for individual processing

        for feat in lyr_tmp:

            geom = feat.GetGeometryRef()

            # filter features that fall within border area
            if border_area.Intersects(geom):
                # filter features that are complete (do not touch border buffer)
                if not buff_geom.Intersects(geom):
                    border_features.append(feat.Clone())

        #border_area.Transform(coordtrans_back)
        #buff_geom.Transform(coordtrans_back)         

        lyr_tmp.ResetReading()

        ##############################################
        # identify fields touching both boundaries on all 4 edges (i.e. large fields)

        for i in range(len(filter_b_ls)):
            
            # set filter to check borders for overlapping large fields
            buffer_lyr.SetAttributeFilter(filter_b_ls[i])
            tile_lyr.SetAttributeFilter(filter_t_ls[i])
                            
            # get relevant border
            # parallel buffer lines
            border_geoms = get_relevant_border(tile_lyr, buffer_lyr)
            
            # loop over fields
            for feat_tmp in lyr_tmp:

                # set to True and overwrite if False
                bool_intersect = True
                geom_tmp = feat_tmp.GetGeometryRef()

                # loop over tile boundaries
                for geom_buff in border_geoms:

                    if not geom_buff.Intersects(geom_tmp):
                        bool_intersect = False
                        break

                if bool_intersect is True:
                    both_intersect_geom.append(feat_tmp.Clone())

            lyr_tmp.ResetReading()    
        
        # reset reading and close ds
        lyr_tmp = None
        ds_tmp = None
        tile_lyr.SetAttributeFilter(None)
        buffer_lyr.SetAttributeFilter(None)
        buffer_lyr.SetSpatialFilter(None)

In [25]:
############################################
# dissolve large fields 

# group intersecting fields
dissolve_groups = group_intersecting_features_from_list(both_intersect_geom)

# process grouped features to dissolve fields
single_diss_fields = []
dissolved_features = []

for group in dissolve_groups:
    # groups with more then one feature are dissolved
    if len(group) > 1:
        features = [feat for feat in group]

        # dissolve fields
        unioned_geom = features[0].GetGeometryRef()

        for feat in features[1:]:

            unioned_geom = unioned_geom.Union(feat.GetGeometryRef())

        # create new feature for dissolved field
        # define CRS
        spatial_ref = osr.SpatialReference()
        spatial_ref.ImportFromEPSG(epsg)

        # create a new feature based on copied layer definition
        new_feature = ogr.Feature(lyr_defn)

        # set geometries
        new_feature.SetGeometry(unioned_geom)

        # create dictionary to store field values of all fields
        names_ls = []

        for i in range(lyr_defn.GetFieldCount()):

            field_defn = lyr_defn.GetFieldDefn(i)
            names_ls.append(field_defn.GetName())

        field_dict = {'fid': [], "area": []}

        for i in names_ls:

            field_dict[i] = []

        # loop over partial fields
        for feat in features:

            field_dict['fid'].append(feat.GetFID())
            field_dict['area'].append(feat.GetGeometryRef().GetArea())

            for name in names_ls:

                field_dict[name].append(feat.GetField(name))
        
        # weighted mean calculation of median_score based on area
        w_mean = np.average(field_dict["median"], weights=field_dict["area"])
        new_feature.SetField("median", w_mean)

        # append new feature to list
        dissolved_features.append(new_feature)

    else:
        # single fields are stored in list
        single_diss_fields.append(list(group)[0].Clone())

print(f"Dissolved fields for {epsg}_{ti}_{tj}")

Dissolved fields for 32636_1_1


In [26]:
#########################################
# group intersecting border features

border_fields = border_features + single_diss_fields

groups = group_intersecting_features_from_list(border_fields)

print(f"Grouped intersecting fields for {epsg}_{ti}_{tj}")

# free memory
del border_fields, border_features, single_diss_fields

##########################################
# process grouped features to extract highest score for each intersection 
# (multiple fields per group as output possible if highest score is a small field)

single_border_fields = []
highest_score_fields = []

for group in groups:
    # groups with more then one feature are filtered using their score ("median")
    if len(group) > 1:
        features = [feat for feat in group]

        while features:
            item_dict = {"feat": [], "score": []}

            for feat in features:
                item_dict["feat"].append(feat)
                item_dict["score"].append(feat.GetField("median"))
            index = np.argmax(item_dict["score"])
            highest_score_feat = item_dict["feat"][index].Clone()
            highest_score_geom = highest_score_feat.GetGeometryRef()
            # copy field with highest score in list
            highest_score_fields.append(highest_score_feat)
            # process remaining fields to ensure that if the field with highest score is small 
            # other fields that do NOT intersect are assessed again
            remaining_feat = []
            for i, feature in enumerate(features):
                geom = feature.GetGeometryRef()
                if not geometries_intersect(highest_score_geom, geom):
                    remaining_feat.append(feature)
                
            features = remaining_feat
    else:
        # single fields are stored in list
        single_border_fields.append(list(group)[0].Clone())

print(f"Selected intersecting fields with highest score for {epsg}_{ti}_{tj}")

del groups

Grouped intersecting fields for 32636_1_1
Selected intersecting fields with highest score for 32636_1_1


In [27]:
############################################
# adjust geometry of slightly intersecting features based on score
# to ensure that there are no overlapping fields in final dataset

combined_fields = highest_score_fields + single_border_fields + dissolved_features

# sort features by score in descending order
combined_fields.sort(key=lambda feature: feature.GetField("median"), reverse=True)

# iterate over features and adjust geometry if features overlap
# higher score features are processed first to ensure that only fields with lower score are adjusted
for i in range(len(combined_fields)):
    feat1 = combined_fields[i]
    geom1 = feat1.GetGeometryRef()
    bbox1 = geom1.GetEnvelope()

    for j in range(i+1, len(combined_fields)):
        feat2 = combined_fields[j]
        geom2 = feat2.GetGeometryRef()
        bbox2 = geom2.GetEnvelope()

        # check if bboxes overlap
        if bbox_intersects(bbox1, bbox2):
            # check if geoms overlap
            if geom1.Intersects(geom2):
                # calculate difference to remove overlap from geom2 (lower score geom)
                new_geom2 = geom2.Difference(geom1)

                # update geometry
                feat2.SetGeometry(new_geom2)

                combined_fields[j] = feat2

print(f"Adjusted geometries of overlapping fields for {epsg}_{ti}_{tj}")

del highest_score_fields, single_border_fields, dissolved_features

Adjusted geometries of overlapping fields for 32636_1_1


In [28]:
#########################################
# create final_features from combined_fields and central_features

final_features = combined_fields + central_features

# save as gpkg
save_gpkg_final([final_features], out_file,
tmp_core_paths[0], f"mozambique_tile_{epsg}_{ti}_{tj}_cleaned")

print(f"Saved processed fields for {epsg}_{ti}_{tj}")

Saved processed fields for 32636_1_1
